# Day 21 — os + sys + pathlib + datetime + time
## Python Standard Library — Part 1

**Coverage:** os, sys, pathlib, datetime, time

**Rule:** Zero external libraries. Everything from the standard library only.

---
## SECTION 1 — `os` Module

The `os` module gives a portable interface to OS functionality. It wraps OS-level system calls so code runs on Linux, macOS, and Windows without changes.

### 1.1 — Working Directory

In [1]:
import os

# getcwd returns the current working directory as a string
current = os.getcwd()
print("Current directory:", current)

# chdir changes the working directory. All relative paths are now resolved from here.
os.chdir('/tmp')
print("After chdir:", os.getcwd())

# Restore
os.chdir(current)
print("Restored:", os.getcwd())

Current directory: c:\Users\shaur\Downloads


FileNotFoundError: [WinError 2] The system cannot find the file specified: '/tmp'

### 1.2 — Listing Directory Contents

In [2]:
# listdir returns a list of names (not full paths) in the given directory
# Hidden files (starting with .) are included
# The list is NOT sorted — order depends on the filesystem
entries = os.listdir('/tmp')
print("Entries in /tmp:", entries[:10])

# To get full paths, combine with join
full_paths = [os.path.join('/tmp', name) for name in entries]
print("First full path:", full_paths[0] if full_paths else 'empty')

# INTERVIEW NOTE:
# os.listdir does NOT recurse into subdirectories.
# Use os.walk for recursive traversal.

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/tmp'

### 1.3 — Creating and Removing Directories

In [3]:
import shutil

# mkdir creates ONE directory. Fails if parent does not exist.
os.mkdir('/tmp/test_single')
print("mkdir created:", os.path.exists('/tmp/test_single'))

# makedirs creates the full path, including any missing parents
# exist_ok=True means no error if the folder already exists
os.makedirs('/tmp/test_parent/child/grandchild', exist_ok=True)
print("makedirs created nested:", os.path.isdir('/tmp/test_parent/child/grandchild'))

# rmdir removes ONE empty directory. Fails if non-empty.
os.rmdir('/tmp/test_single')
print("rmdir removed:", not os.path.exists('/tmp/test_single'))

# remove deletes a FILE (not a directory)
with open('/tmp/test_file.txt', 'w') as f:
    f.write('hello')
os.remove('/tmp/test_file.txt')
print("remove deleted file:", not os.path.exists('/tmp/test_file.txt'))

# rename renames or moves a file or directory
os.makedirs('/tmp/rename_test', exist_ok=True)
os.rename('/tmp/rename_test', '/tmp/renamed_folder')
print("rename worked:", os.path.exists('/tmp/renamed_folder'))

# Cleanup
os.rmdir('/tmp/renamed_folder')
shutil.rmtree('/tmp/test_parent', ignore_errors=True)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/tmp/test_single'

### 1.4 — os.path — Path Inspection and Manipulation

`os.path` is a module of string-manipulation functions for file paths. It does NOT require the path to actually exist on disk.

In [4]:
path = '/home/user/projects/script.py'

# exists — True if path exists (file OR directory)
print("exists:", os.path.exists(path))           # False

# isfile — True only if it is a regular file
print("isfile:", os.path.isfile(path))

# isdir — True only if it is a directory
print("isdir /tmp:", os.path.isdir('/tmp'))

# join — joins path components using the OS separator
# ALWAYS use join instead of string concatenation for paths
joined = os.path.join('/home/user', 'projects', 'script.py')
print("join:", joined)

# basename — last component of the path
print("basename:", os.path.basename(path))        # script.py

# dirname — everything except the last component
print("dirname:", os.path.dirname(path))          # /home/user/projects

# splitext — splits into (root, extension)
root, ext = os.path.splitext(path)
print("splitext root:", root)                     # /home/user/projects/script
print("splitext ext:", ext)                       # .py

# abspath — converts relative path to absolute using getcwd as base
print("abspath:", os.path.abspath('notes.txt'))

# INTERVIEW GOTCHA:
# os.path.join('/tmp', '/absolute') returns '/absolute'
# An absolute component discards everything before it
print("gotcha:", os.path.join('/tmp', '/absolute'))   # /absolute

exists: False
isfile: False
isdir /tmp: False
join: /home/user\projects\script.py
basename: script.py
dirname: /home/user/projects
splitext root: /home/user/projects/script
splitext ext: .py
abspath: c:\Users\shaur\Downloads\notes.txt
gotcha: /absolute


### 1.5 — Environment Variables

In [5]:
# os.environ is a dictionary-like mapping of environment variables
# Changes affect the current process and its children, NOT the parent shell

# get — safe access with a default (avoids KeyError if key missing)
home = os.environ.get('HOME', '/default/home')
print("HOME:", home)

path_var = os.environ.get('PATH', '')
print("PATH (first 80 chars):", path_var[:80])

# set — add or overwrite a variable
os.environ['MY_APP_ENV'] = 'production'
print("Set MY_APP_ENV:", os.environ.get('MY_APP_ENV'))

# delete
del os.environ['MY_APP_ENV']
print("After delete:", os.environ.get('MY_APP_ENV'))   # None

print("Total env vars:", len(os.environ))

# INTERVIEW NOTE:
# In production, always use os.environ.get() instead of os.environ[]
# because os.environ[] raises KeyError if the variable is missing.

HOME: /default/home
PATH (first 80 chars): c:\Users\shaur\AppData\Local\Programs\Python\Python313;c:\Users\shaur\AppData\Ro
Set MY_APP_ENV: production
After delete: None
Total env vars: 77


### 1.6 — os.walk — Recursive Directory Traversal

`os.walk` is a generator that yields `(dirpath, dirnames, filenames)` tuples for every directory in a tree.

In [6]:
import os, shutil

# Create a test tree
os.makedirs('/tmp/walk_demo/subA', exist_ok=True)
os.makedirs('/tmp/walk_demo/subB', exist_ok=True)
for name in ['file1.txt', 'file2.py', 'subA/a.txt', 'subB/b.csv']:
    with open(f'/tmp/walk_demo/{name}', 'w') as f:
        f.write('demo')

# os.walk — top-down by default
# dirpath  = current folder being visited (as string)
# dirnames = list of subdirectory names inside dirpath
# filenames = list of file names inside dirpath
print("--- os.walk demo ---")
for dirpath, dirnames, filenames in os.walk('/tmp/walk_demo'):
    level = dirpath.replace('/tmp/walk_demo', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}DIR: {dirpath}")
    for fname in filenames:
        print(f"{indent}  FILE: {fname}")

# Collecting all .txt files recursively — real pattern used in projects
txt_files = []
for dirpath, _, filenames in os.walk('/tmp/walk_demo'):
    for fname in filenames:
        if fname.endswith('.txt'):
            txt_files.append(os.path.join(dirpath, fname))
print("\nAll .txt files:", txt_files)

# bottomup=True — children visited before parents
# Useful when deleting trees (must remove children first)

shutil.rmtree('/tmp/walk_demo')

--- os.walk demo ---
DIR: /tmp/walk_demo
  FILE: file1.txt
  FILE: file2.py
  DIR: /tmp/walk_demo\subA
    FILE: a.txt
  DIR: /tmp/walk_demo\subB
    FILE: b.csv

All .txt files: ['/tmp/walk_demo\\file1.txt', '/tmp/walk_demo\\subA\\a.txt']


### 1.7 — Process Information

In [7]:
# os.system runs a shell command — returns exit code (0 = success)
# AVOID in production — use subprocess.run() instead (safer, no shell injection)
exit_code = os.system('echo "os.system works"')
print("Exit code:", exit_code)

# getpid — process ID of the current Python process
print("PID:", os.getpid())

# cpu_count — number of logical CPUs
# Used for setting worker count in ThreadPoolExecutor or multiprocessing.Pool
print("CPU count:", os.cpu_count())

# INTERVIEW NOTE:
# os.cpu_count() can return None if count cannot be determined.
# Safe pattern: workers = os.cpu_count() or 4

Exit code: 0
PID: 12000
CPU count: 8


---
## SECTION 2 — `sys` Module

`sys` gives access to variables and functions maintained by the Python interpreter itself. It controls interpreter behavior at runtime.

### 2.1 — sys.argv — Command Line Arguments

In [8]:
import sys

# sys.argv is a list of strings
# argv[0] = name of the script being run
# argv[1], argv[2], ... = arguments passed by the user on the command line

# In a real script run as: python organizer.py /home/user/Downloads
# sys.argv would be: ['organizer.py', '/home/user/Downloads']

print("sys.argv:", sys.argv)
print("Script name:", sys.argv[0])

# Safe extraction with a default
target_folder = sys.argv[1] if len(sys.argv) > 1 else '.'
print("Target folder:", target_folder)

# Standard validation pattern in CLI scripts:
# if len(sys.argv) < 2:
#     print("Usage: python script.py <folder>", file=sys.stderr)
#     sys.exit(1)

print("\nINTERVIEW NOTE: sys.argv[0] is always the script path.")
print("User-provided args start at sys.argv[1].")

sys.argv: ['C:\\Users\\shaur\\AppData\\Roaming\\Python\\Python313\\site-packages\\ipykernel_launcher.py', '--f=c:\\Users\\shaur\\AppData\\Roaming\\jupyter\\runtime\\kernel-v33557c797dfc24cb0c9468e1abc6c53b4eeeb7685.json']
Script name: C:\Users\shaur\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py
Target folder: --f=c:\Users\shaur\AppData\Roaming\jupyter\runtime\kernel-v33557c797dfc24cb0c9468e1abc6c53b4eeeb7685.json

INTERVIEW NOTE: sys.argv[0] is always the script path.
User-provided args start at sys.argv[1].


### 2.2 — sys.path — Module Search Path

In [9]:
# sys.path is a list of directories Python searches when you do import
# Python checks them in order — first match wins

print("sys.path entries:")
for p in sys.path:
    print(" ", p)

# append — add to the END of the search path
sys.path.append('/tmp/my_custom_modules')
print("\nAfter append, last entry:", sys.path[-1])

# insert(0, ...) means check this folder FIRST before anything else
sys.path.insert(0, '/tmp/priority_modules')
print("After insert(0), first entry:", sys.path[0])

# Clean up test entries
sys.path.pop(0)
sys.path.pop()

# INTERVIEW NOTE:
# Modifying sys.path at runtime is a workaround, not best practice.
# Proper solution: use packages with __init__.py or install via pip.

sys.path entries:
  c:\Users\shaur\AppData\Local\Programs\Python\Python313\python313.zip
  c:\Users\shaur\AppData\Local\Programs\Python\Python313\DLLs
  c:\Users\shaur\AppData\Local\Programs\Python\Python313\Lib
  c:\Users\shaur\AppData\Local\Programs\Python\Python313
  
  C:\Users\shaur\AppData\Roaming\Python\Python313\site-packages
  C:\Users\shaur\AppData\Roaming\Python\Python313\site-packages\win32
  C:\Users\shaur\AppData\Roaming\Python\Python313\site-packages\win32\lib
  C:\Users\shaur\AppData\Roaming\Python\Python313\site-packages\Pythonwin
  c:\Users\shaur\AppData\Local\Programs\Python\Python313\Lib\site-packages

After append, last entry: /tmp/my_custom_modules
After insert(0), first entry: /tmp/priority_modules


'/tmp/my_custom_modules'

### 2.3 — Exit, Streams, and Interpreter Info

In [10]:
# sys.exit raises SystemExit exception
# exit(0) = success, exit(1) = error, exit(string) = print to stderr then exit
# Can be caught with try/except SystemExit
try:
    sys.exit(0)
except SystemExit as e:
    print("Caught SystemExit with code:", e.code)

# sys.stdout — standard output stream (what print() writes to)
sys.stdout.write("Written directly to stdout\n")

# sys.stderr — standard error stream
# Errors go here. Stays visible even when stdout is redirected to a file.
sys.stderr.write("This is an error message\n")

# Using print to write to stderr
print("Error via print", file=sys.stderr)

# Interpreter information
print("\nPython version:", sys.version)
print("Platform:", sys.platform)        # 'linux', 'darwin', 'win32'
print("Executable:", sys.executable)    # Full path to python binary
print("Max int size:", sys.maxsize)     # Largest int on this platform

# getsizeof — memory size of an object in bytes (SHALLOW measurement)
print("\ngetsizeof(42):", sys.getsizeof(42))
print("getsizeof([]):", sys.getsizeof([]))
print("getsizeof('hello'):", sys.getsizeof('hello'))

# INTERVIEW GOTCHA:
# getsizeof is SHALLOW — does not count objects contained inside a list/dict
# Use pympler or tracemalloc for deep memory profiling

# sys.modules — dictionary of all currently imported modules
print("\nTotal loaded modules:", len(sys.modules))
print("'os' in sys.modules:", 'os' in sys.modules)

Caught SystemExit with code: 0
Written directly to stdout

Python version: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
Platform: win32
Executable: c:\Users\shaur\AppData\Local\Programs\Python\Python313\python.exe
Max int size: 9223372036854775807

getsizeof(42): 28
getsizeof([]): 56
getsizeof('hello'): 46

Total loaded modules: 938
'os' in sys.modules: True


This is an error message
Error via print


---
## SECTION 3 — `pathlib` Module

`pathlib` (Python 3.4+) replaces `os.path` with an object-oriented interface. Instead of passing strings, you work with `Path` objects that have methods and properties.

On Windows you get `WindowsPath`, on Linux/macOS you get `PosixPath`. Both behave identically through the `Path` interface.

### 3.1 — Creating Path Objects

In [11]:
from pathlib import Path

# Create from string
p = Path('/home/user/projects/script.py')
print("Path:", p)
print("Type:", type(p))

# Current directory
cwd = Path.cwd()
print("cwd:", cwd)

# Home directory of current user
home = Path.home()
print("home:", home)

# Relative path (no leading slash)
rel = Path('data/input.csv')
print("relative:", rel)
print("is_absolute:", rel.is_absolute())

# absolute — resolves relative to cwd (does not resolve symlinks)
# resolve  — like absolute but also resolves symlinks
print("absolute:", rel.absolute())
print("resolve:", rel.resolve())

Path: \home\user\projects\script.py
Type: <class 'pathlib._local.WindowsPath'>
cwd: c:\Users\shaur\Downloads
home: C:\Users\shaur
relative: data\input.csv
is_absolute: False
absolute: c:\Users\shaur\Downloads\data\input.csv
resolve: C:\Users\shaur\Downloads\data\input.csv


### 3.2 — The `/` Operator and joinpath

In [12]:
base = Path('/home/user')

# / operator — the cleanest way to join paths
# This overloads Python's real division operator via __truediv__
joined = base / 'projects' / 'script.py'
print("/ operator:", joined)

# joinpath — equivalent to / but method-style, accepts multiple args
joined2 = base.joinpath('projects', 'script.py')
print("joinpath:", joined2)

print("Equal:", joined == joined2)

# INTERVIEW NOTE:
# Path('/a') / 'b' calls Path.__truediv__('b')
# Same gotcha as os.path.join: absolute segment discards everything before it
print("Gotcha:", Path('/home/user') / Path('/tmp'))   # /tmp

/ operator: \home\user\projects\script.py
joinpath: \home\user\projects\script.py
Equal: True
Gotcha: \tmp


### 3.3 — Path Properties

In [13]:
p = Path('/home/user/projects/my_script.py')

print("name:", p.name)             # my_script.py     — filename with extension
print("stem:", p.stem)             # my_script         — filename without extension
print("suffix:", p.suffix)         # .py               — extension with dot

# suffixes — all extensions for multi-extension files
archive = Path('backup.tar.gz')
print("suffixes:", archive.suffixes)   # ['.tar', '.gz']
print("suffix:", archive.suffix)       # .gz — only the LAST one

print("parent:", p.parent)         # /home/user/projects — one level up

print("parents:")
for ancestor in p.parents:         # all ancestors
    print(" ", ancestor)

print("parts:", p.parts)           # ('/', 'home', 'user', 'projects', 'my_script.py')
print("anchor:", p.anchor)         # / on Unix, C:\ on Windows

# as_posix — convert to forward-slash string (useful for cross-platform code)
print("as_posix:", p.as_posix())

name: my_script.py
stem: my_script
suffix: .py
suffixes: ['.tar', '.gz']
suffix: .gz
parent: \home\user\projects
parents:
  \home\user\projects
  \home\user
  \home
  \
parts: ('\\', 'home', 'user', 'projects', 'my_script.py')
anchor: \
as_posix: /home/user/projects/my_script.py


### 3.4 — Checking Existence and Type

In [14]:
tmp = Path('/tmp')
fake = Path('/tmp/does_not_exist_xyz.txt')

print("exists /tmp:", tmp.exists())        # True
print("exists fake:", fake.exists())       # False

print("is_file /tmp:", tmp.is_file())      # False
print("is_dir /tmp:", tmp.is_dir())        # True

print("is_absolute:", tmp.is_absolute())   # True
print("relative is_absolute:", Path('data').is_absolute())   # False

exists /tmp: True
exists fake: False
is_file /tmp: False
is_dir /tmp: True
is_absolute: False
relative is_absolute: False


### 3.5 — Creating, Deleting, Renaming

In [15]:
# mkdir — parents=True creates missing parents, exist_ok avoids error if exists
new_dir = Path('/tmp/pathlib_demo')
new_dir.mkdir(parents=True, exist_ok=True)
print("mkdir created:", new_dir.exists())

# touch — creates an empty file, or updates modification time if already exists
new_file = new_dir / 'notes.txt'
new_file.touch()
print("touch created:", new_file.exists())

# rename — moves and renames the file; returns the new Path object
renamed = new_file.rename(new_dir / 'renamed_notes.txt')
print("renamed:", renamed.name)

# unlink — deletes a FILE
# missing_ok=True (Python 3.8+) means no error if already gone
renamed.unlink()
print("unlink deleted:", not renamed.exists())

# rmdir — deletes an EMPTY directory
new_dir.rmdir()
print("rmdir deleted:", not new_dir.exists())

# INTERVIEW NOTE:
# To delete non-empty directories, use shutil.rmtree(path)
# Path.rmdir() only works on empty directories

mkdir created: True
touch created: True
renamed: renamed_notes.txt
unlink deleted: True
rmdir deleted: True


### 3.6 — Reading and Writing Files

In [16]:
test_file = Path('/tmp/pathlib_test.txt')

# write_text — write a string to a file (creates or overwrites)
test_file.write_text('Hello from pathlib\nLine two\n', encoding='utf-8')

# read_text — read entire file as a single string
content = test_file.read_text(encoding='utf-8')
print("read_text:", repr(content))

# write_bytes / read_bytes — for binary data
bin_file = Path('/tmp/pathlib_test.bin')
bin_file.write_bytes(b'\x00\x01\x02\xFF')
data = bin_file.read_bytes()
print("read_bytes:", data)

# open — same as built-in open() but called as a method on the Path object
with test_file.open('r', encoding='utf-8') as f:
    first_line = f.readline()
print("First line via open:", repr(first_line))

# Cleanup
test_file.unlink()
bin_file.unlink()

# INTERVIEW NOTE:
# path.read_text() is equivalent to:
#   with open(path, 'r') as f: return f.read()
# It is convenience, not more powerful.

read_text: 'Hello from pathlib\nLine two\n'
read_bytes: b'\x00\x01\x02\xff'
First line via open: 'Hello from pathlib\n'


### 3.7 — Globbing and Iteration

In [17]:
import shutil
base = Path('/tmp/glob_demo')
base.mkdir(exist_ok=True)
(base / 'sub').mkdir(exist_ok=True)
for name in ['a.py', 'b.py', 'c.txt', 'sub/d.py', 'sub/e.csv']:
    (base / name).write_text('x')

# iterdir — yields direct children (files and folders) — non-recursive
print("iterdir:")
for item in sorted(base.iterdir()):
    print(" ", item.name, '(dir)' if item.is_dir() else '(file)')

# glob — match files using a shell pattern — non-recursive by default
# * matches any characters within ONE path component
print("\nglob '*.py' (top level only):")
for f in sorted(base.glob('*.py')):
    print(" ", f.name)

# rglob — recursive glob — searches all subdirectories
# rglob('*.py') is exactly equivalent to glob('**/*.py')
print("\nrglob '*.py' (all subdirs):")
for f in sorted(base.rglob('*.py')):
    print(" ", f.relative_to(base))

# ** in glob pattern — matches any number of directory levels
print("\nglob '**/*.csv':")
for f in base.glob('**/*.csv'):
    print(" ", f)

shutil.rmtree(base)

# INTERVIEW NOTE:
# glob and rglob return generators, not lists.
# Wrap in list() or sorted() to consume them all at once.
# Generators can only be iterated ONCE.

iterdir:
  a.py (file)
  b.py (file)
  c.txt (file)
  sub (dir)

glob '*.py' (top level only):
  a.py
  b.py

rglob '*.py' (all subdirs):
  a.py
  b.py
  sub\d.py

glob '**/*.csv':
  \tmp\glob_demo\sub\e.csv


---
## SECTION 4 — `datetime` Module

Four main classes:
- `date` — year, month, day
- `time` — hour, minute, second, microsecond
- `datetime` — date + time combined
- `timedelta` — duration (difference between two datetimes)

### 4.1 — date Object

In [18]:
from datetime import date, time, datetime, timedelta

# Create a date object
d = date(2025, 1, 15)
print("date:", d)
print("year:", d.year)
print("month:", d.month)
print("day:", d.day)

today = date.today()
print("today:", today)

# weekday — Monday=0, Sunday=6
print("weekday:", today.weekday())

# isoweekday — Monday=1, Sunday=7
print("isoweekday:", today.isoweekday())

# isoformat — returns 'YYYY-MM-DD' string
print("isoformat:", today.isoformat())

# fromisoformat — parse 'YYYY-MM-DD' string
parsed = date.fromisoformat('2025-07-04')
print("fromisoformat:", parsed)

date: 2025-01-15
year: 2025
month: 1
day: 15
today: 2026-05-02
weekday: 5
isoweekday: 6
isoformat: 2026-05-02
fromisoformat: 2025-07-04


### 4.2 — datetime Object

In [19]:
from datetime import timezone

# Create a datetime object
dt = datetime(2025, 6, 15, 10, 30, 45)
print("datetime:", dt)
print("date part:", dt.date())
print("time part:", dt.time())
print("hour:", dt.hour)
print("minute:", dt.minute)
print("second:", dt.second)

# now — current date and time in local time, no timezone info (naive)
now = datetime.now()
print("\nnow:", now)

# The correct way to get current UTC time (timezone-aware)
now_utc = datetime.now(timezone.utc)
print("now UTC:", now_utc)

# isoformat
print("isoformat:", now.isoformat())

# fromisoformat — parse ISO 8601 string
parsed_dt = datetime.fromisoformat('2025-06-15T10:30:45')
print("fromisoformat:", parsed_dt)

datetime: 2025-06-15 10:30:45
date part: 2025-06-15
time part: 10:30:45
hour: 10
minute: 30
second: 45

now: 2026-05-02 23:08:28.979598
now UTC: 2026-05-02 17:23:28.979695+00:00
isoformat: 2026-05-02T23:08:28.979598
fromisoformat: 2025-06-15 10:30:45


### 4.3 — strptime and strftime — Parsing and Formatting

In [20]:
dt = datetime(2025, 6, 15, 14, 30, 0)

# strftime — FORMAT a datetime into a string
# Memory trick: f = format (output is a formatted string)
print("strftime examples:")
print(dt.strftime('%Y-%m-%d'))             # 2025-06-15
print(dt.strftime('%d/%m/%Y'))             # 15/06/2025
print(dt.strftime('%H:%M:%S'))             # 14:30:00
print(dt.strftime('%I:%M %p'))             # 02:30 PM  (%I = 12-hour clock)
print(dt.strftime('%A, %B %d, %Y'))        # Sunday, June 15, 2025
print(dt.strftime('%a %b %d %Y'))          # Sun Jun 15 2025

# strptime — PARSE a string into a datetime object
# Memory trick: p = parse (input is a string to parse)
# The format string MUST match the input string exactly
date_str = '15-Jun-2025 14:30'
parsed = datetime.strptime(date_str, '%d-%b-%Y %H:%M')
print("\nstrptime result:", parsed)

print("\nFormat code reference:")
codes = [
    ('%Y', '4-digit year',          '2025'),
    ('%m', 'zero-padded month',     '06'),
    ('%d', 'zero-padded day',       '05'),
    ('%H', '24-hour hour',          '14'),
    ('%M', 'minute',                '30'),
    ('%S', 'second',                '00'),
    ('%A', 'full weekday name',     'Monday'),
    ('%a', 'abbreviated weekday',   'Mon'),
    ('%B', 'full month name',       'June'),
    ('%b', 'abbreviated month',     'Jun'),
]
for code, desc, example in codes:
    print(f"  {code}  {desc:<25} e.g. {example}")

# INTERVIEW NOTE:
# strptime raises ValueError if the string does not match the format.
# Always wrap in try/except when parsing user-provided date strings.

strftime examples:
2025-06-15
15/06/2025
14:30:00
02:30 PM
Sunday, June 15, 2025
Sun Jun 15 2025

strptime result: 2025-06-15 14:30:00

Format code reference:
  %Y  4-digit year              e.g. 2025
  %m  zero-padded month         e.g. 06
  %d  zero-padded day           e.g. 05
  %H  24-hour hour              e.g. 14
  %M  minute                    e.g. 30
  %S  second                    e.g. 00
  %A  full weekday name         e.g. Monday
  %a  abbreviated weekday       e.g. Mon
  %B  full month name           e.g. June
  %b  abbreviated month         e.g. Jun


### 4.4 — timedelta — Duration Arithmetic

In [21]:
from datetime import timedelta

# timedelta represents a DURATION, not a point in time
# Internally stored as (days, seconds, microseconds)

delta = timedelta(days=10, hours=5, minutes=30)
print("timedelta:", delta)
print("days:", delta.days)
print("seconds:", delta.seconds)             # Only the seconds component
print("total_seconds:", delta.total_seconds())   # TOTAL duration in seconds

# ARITHMETIC — add/subtract timedelta from datetime
today = datetime.now()

deadline = today + timedelta(days=30)
print("\n30 days from now:", deadline.strftime('%Y-%m-%d'))

last_week = today - timedelta(weeks=1)
print("1 week ago:", last_week.strftime('%Y-%m-%d'))

# Difference between two datetimes gives a timedelta
start = datetime(2025, 1, 1)
end   = datetime(2025, 12, 31)
diff  = end - start
print("\nDays in 2025:", diff.days)

# Days remaining until a target
target = datetime(2025, 12, 31)
remaining = target - datetime.now()
print("Days until Dec 31:", remaining.days)
print("Overdue?:", remaining.days < 0)

# INTERVIEW GOTCHA:
# timedelta.seconds is NOT total_seconds()
d = timedelta(days=1, seconds=30)
print("\nd.seconds:", d.seconds)              # 30 — just the seconds component
print("d.total_seconds():", d.total_seconds())  # 86430.0 = 86400 + 30

timedelta: 10 days, 5:30:00
days: 10
seconds: 19800
total_seconds: 883800.0

30 days from now: 2026-06-01
1 week ago: 2026-04-25

Days in 2025: 364
Days until Dec 31: -123
Overdue?: True

d.seconds: 30
d.total_seconds(): 86430.0


### 4.5 — Unix Timestamps and Timezones

In [22]:
from datetime import datetime, timezone, timedelta
import time

# UNIX TIMESTAMP — seconds since Jan 1, 1970 00:00:00 UTC
unix_now = time.time()
print("Unix timestamp now:", unix_now)

# Convert Unix timestamp to datetime
dt_local = datetime.fromtimestamp(unix_now)                       # local time (naive)
dt_utc   = datetime.fromtimestamp(unix_now, tz=timezone.utc)     # UTC (aware)
print("from timestamp (local):", dt_local)
print("from timestamp (UTC):", dt_utc)

# Convert datetime to Unix timestamp
ts = datetime.now().timestamp()
print("timestamp():", ts)

# NAIVE vs AWARE datetimes
naive = datetime.now()
print("\nNaive tzinfo:", naive.tzinfo)       # None

aware_utc = datetime.now(timezone.utc)
print("Aware tzinfo:", aware_utc.tzinfo)

# Nepal Standard Time: UTC+5:45
nepal_tz = timezone(timedelta(hours=5, minutes=45))
aware_npt = datetime.now(nepal_tz)
print("Nepal time:", aware_npt.strftime('%Y-%m-%d %H:%M %Z'))

# astimezone — convert between timezones
ist_tz = timezone(timedelta(hours=5, minutes=30))
aware_ist = aware_utc.astimezone(ist_tz)
print("IST time:", aware_ist.strftime('%Y-%m-%d %H:%M %Z'))

# INTERVIEW NOTE:
# You CANNOT compare naive and aware datetimes — raises TypeError.
# Always store datetimes in UTC, convert to local only for display.
try:
    diff = aware_utc - naive
except TypeError as e:
    print("\nComparing naive and aware:", e)

Unix timestamp now: 1777742610.4712684
from timestamp (local): 2026-05-02 23:08:30.471268
from timestamp (UTC): 2026-05-02 17:23:30.471268+00:00
timestamp(): 1777742610.471732

Naive tzinfo: None
Aware tzinfo: UTC
Nepal time: 2026-05-02 23:08 UTC+05:45
IST time: 2026-05-02 22:53 UTC+05:30

Comparing naive and aware: can't subtract offset-naive and offset-aware datetimes


---
## SECTION 5 — `time` Module

The `time` module provides low-level access to time-related system calls.
Main uses: benchmarking, sleeping, raw timestamps, and struct_time manipulation.

### 5.1 — Clock Functions

In [23]:
import time

# time.time() — Unix timestamp (float)
# Affected by system clock adjustments (NTP sync, DST, manual changes)
# USE FOR: timestamping events, working with real-world time
print("time.time():", time.time())

# time.perf_counter() — high-resolution performance counter
# NOT affected by system clock changes
# Includes time spent sleeping
# USE FOR: benchmarking code, measuring elapsed wall time
print("time.perf_counter():", time.perf_counter())

# time.monotonic() — always increases, never goes backward
# Like perf_counter but may have lower resolution
# USE FOR: timeout logic where you need guaranteed forward movement
print("time.monotonic():", time.monotonic())

# time.process_time() — CPU time used by THIS process only
# Does NOT count time spent sleeping
# USE FOR: measuring CPU usage, not wall-clock time
print("time.process_time():", time.process_time())

print("\nClock comparison:")
print("  time()          — real world Unix timestamp, CAN go backward")
print("  perf_counter()  — best for benchmarking, includes sleep time")
print("  monotonic()     — never decreases, good for timeouts")
print("  process_time()  — CPU time only, excludes sleep")

time.time(): 1777742611.8779497
time.perf_counter(): 6288.400172
time.monotonic(): 6288.4002679
time.process_time(): 1.75

Clock comparison:
  time()          — real world Unix timestamp, CAN go backward
  perf_counter()  — best for benchmarking, includes sleep time
  monotonic()     — never decreases, good for timeouts
  process_time()  — CPU time only, excludes sleep


### 5.2 — Execution Time Pattern (Benchmarking)

In [24]:
# STANDARD BENCHMARKING PATTERN — use perf_counter
start = time.perf_counter()

# Code being benchmarked
total = sum(range(1_000_000))

end = time.perf_counter()
elapsed = end - start

print(f"sum(range(1_000_000)) = {total}")
print(f"Elapsed: {elapsed:.6f} seconds")
print(f"Elapsed: {elapsed * 1000:.3f} ms")

# For microbenchmarks (very fast code), use the timeit module instead.
# timeit runs the code N times and averages the results.
import timeit
result = timeit.timeit('sum(range(1000))', number=10_000)
print(f"\ntimeit: {result:.4f}s for 10,000 runs")
print(f"Per run: {result / 10_000 * 1_000:.4f} ms")

# INTERVIEW NOTE:
# perf_counter is best for macro-benchmarks (> ~10ms operations).
# For sub-millisecond operations, timeit is more accurate because
# it runs many repetitions and reduces measurement noise.

sum(range(1_000_000)) = 499999500000
Elapsed: 0.014910 seconds
Elapsed: 14.910 ms

timeit: 0.1131s for 10,000 runs
Per run: 0.0113 ms


### 5.3 — sleep, localtime, gmtime, strftime, mktime

In [25]:
# sleep — pause execution for N seconds
# Accepts float: sleep(0.5) = 500 milliseconds
print("Sleeping 0.1 seconds...")
start = time.perf_counter()
time.sleep(0.1)
elapsed = time.perf_counter() - start
print(f"Actual sleep: {elapsed:.4f}s")

# INTERVIEW NOTE: time.sleep releases the GIL during the sleep.
# Other threads CAN run while one thread is sleeping.

# localtime — converts Unix timestamp to struct_time in LOCAL timezone
ts = time.time()
local = time.localtime(ts)
print("\nlocaltime:", local)
print("Year:", local.tm_year)
print("Month:", local.tm_mon)
print("Hour:", local.tm_hour)

# gmtime — same but always UTC regardless of system timezone
gmt = time.gmtime(ts)
print("\ngmtime UTC hour:", gmt.tm_hour)

# time.strftime — format a struct_time into a string
# Works on struct_time objects (from localtime/gmtime), not datetime objects
formatted = time.strftime('%Y-%m-%d %H:%M:%S', local)
print("time.strftime:", formatted)

# mktime — converts a LOCAL struct_time back to a Unix timestamp
back_to_ts = time.mktime(local)
print("mktime:", back_to_ts)
print("Round-trip matches:", abs(back_to_ts - ts) < 1)

Sleeping 0.1 seconds...
Actual sleep: 0.1005s

localtime: time.struct_time(tm_year=2026, tm_mon=5, tm_mday=2, tm_hour=23, tm_min=8, tm_sec=33, tm_wday=5, tm_yday=122, tm_isdst=0)
Year: 2026
Month: 5
Hour: 23

gmtime UTC hour: 17
time.strftime: 2026-05-02 23:08:33
mktime: 1777742613.0
Round-trip matches: True


### 5.4 — get_clock_info and Timezone Info

In [26]:
# get_clock_info — returns metadata about each clock
print("Clock metadata:")
for clock_name in ['time', 'perf_counter', 'monotonic', 'process_time']:
    info = time.get_clock_info(clock_name)
    print(f"\n{clock_name}:")
    print(f"  adjustable     = {info.adjustable}")      # Can the OS change this clock?
    print(f"  implementation = {info.implementation}")   # Underlying system call used
    print(f"  monotonic      = {info.monotonic}")       # Does it never go backward?
    print(f"  resolution     = {info.resolution}")      # Smallest measurable unit (seconds)

# Timezone info from the time module
print("\ntime.timezone:", time.timezone)   # UTC offset in seconds (negative = east of UTC)
print("time.altzone:", time.altzone)       # DST offset in seconds
print("time.tzname:", time.tzname)         # ('Standard name', 'DST name')
print("time.daylight:", time.daylight)     # 1 if DST is observed in this locale

Clock metadata:

time:
  adjustable     = True
  implementation = GetSystemTimePreciseAsFileTime()
  monotonic      = False
  resolution     = 1e-07

perf_counter:
  adjustable     = False
  implementation = QueryPerformanceCounter()
  monotonic      = True
  resolution     = 1e-07

monotonic:
  adjustable     = False
  implementation = QueryPerformanceCounter()
  monotonic      = True
  resolution     = 1e-07

process_time:
  adjustable     = False
  implementation = GetProcessTimes()
  monotonic      = True
  resolution     = 1e-07

time.timezone: -20700
time.altzone: -24300
time.tzname: ('Nepal Standard Time', 'Nepal Daylight Time')
time.daylight: 0


---
## SECTION 6 — GitHub Project: file_system_tool.py
### Smart File Organizer

What it does:
1. Takes a folder path from sys.argv
2. Scans all files recursively using os.walk
3. Groups files by extension
4. Creates category subdirectories using pathlib
5. Moves files into the correct category folder
6. Logs every action with a datetime timestamp

In [27]:
import os, sys, shutil, logging
from pathlib import Path
from datetime import datetime

EXTENSION_MAP = {
    '.pdf':  'PDFs',    '.doc':  'Documents', '.docx': 'Documents',
    '.txt':  'Documents', '.jpg':  'Images',   '.jpeg': 'Images',
    '.png':  'Images',  '.gif':  'Images',    '.py':   'Scripts',
    '.js':   'Scripts', '.sh':   'Scripts',   '.mp3':  'Audio',
    '.wav':  'Audio',   '.mp4':  'Videos',    '.mov':  'Videos',
    '.zip':  'Archives','.tar':  'Archives',  '.gz':   'Archives',
}

def setup_logging(log_dir: Path) -> None:
    log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'organizer_{timestamp}.log'
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s | %(levelname)s | %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout),
        ]
    )
    logging.info(f"Log file: {log_file}")

def get_category(file_path: Path) -> str:
    ext = file_path.suffix.lower()
    return EXTENSION_MAP.get(ext, 'Others')

def collect_files(source_dir: Path) -> list:
    files = []
    category_names = set(EXTENSION_MAP.values())
    for dirpath, dirnames, filenames in os.walk(source_dir):
        current_dir = Path(dirpath)
        # Skip category folders so we do not re-organize already-organized files
        if current_dir.name in category_names:
            dirnames.clear()
            continue
        for fname in filenames:
            files.append(current_dir / fname)
    logging.info(f"Found {len(files)} files to organize")
    return files

def organize(source_dir: Path) -> dict:
    summary = {}
    if not source_dir.exists() or not source_dir.is_dir():
        logging.error(f"Invalid directory: {source_dir}")
        return summary
    for file_path in collect_files(source_dir):
        category = get_category(file_path)
        dest_dir  = source_dir / category
        dest_file = dest_dir / file_path.name
        # Handle name collision by appending a counter
        counter = 1
        while dest_file.exists():
            dest_file = dest_dir / f"{file_path.stem}_{counter}{file_path.suffix}"
            counter += 1
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.move(str(file_path), str(dest_file))
        logging.info(
            f"Moved: {file_path.name} -> {category}/ "
            f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]"
        )
        summary[category] = summary.get(category, 0) + 1
    return summary

def print_summary(summary: dict) -> None:
    logging.info("--- Organization Summary ---")
    for category, count in sorted(summary.items()):
        logging.info(f"  {category:<15} {count:>4} file(s)")
    logging.info(f"  {'TOTAL':<15} {sum(summary.values()):>4} file(s)")

# --- DEMO ---
# Reload logging without duplicate handlers for notebook environment
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

demo_dir = Path('/tmp/demo_organizer')
demo_dir.mkdir(exist_ok=True)
for fname in ['report.pdf', 'photo.jpg', 'script.py', 'notes.txt',
              'video.mp4', 'data.csv', 'backup.zip']:
    (demo_dir / fname).write_text('demo content')

setup_logging(demo_dir / 'logs')
logging.info(f"Organizing: {demo_dir}")

summary = organize(demo_dir)
print_summary(summary)

print("\nFinal structure:")
for item in sorted(demo_dir.iterdir()):
    if item.is_dir() and item.name != 'logs':
        print(f"  {item.name}/")
        for f in sorted(item.iterdir()):
            print(f"    {f.name}")

shutil.rmtree(demo_dir)

2026-05-02 23:08:33,842 | INFO | Log file: \tmp\demo_organizer\logs\organizer_20260502_230833.log
2026-05-02 23:08:33,843 | INFO | Organizing: \tmp\demo_organizer
2026-05-02 23:08:33,845 | INFO | Found 8 files to organize
2026-05-02 23:08:33,848 | INFO | Moved: backup.zip -> Archives/ [2026-05-02 23:08:33]
2026-05-02 23:08:33,851 | INFO | Moved: data.csv -> Others/ [2026-05-02 23:08:33]
2026-05-02 23:08:33,854 | INFO | Moved: notes.txt -> Documents/ [2026-05-02 23:08:33]
2026-05-02 23:08:33,856 | INFO | Moved: photo.jpg -> Images/ [2026-05-02 23:08:33]
2026-05-02 23:08:33,861 | INFO | Moved: report.pdf -> PDFs/ [2026-05-02 23:08:33]
2026-05-02 23:08:33,864 | INFO | Moved: script.py -> Scripts/ [2026-05-02 23:08:33]
2026-05-02 23:08:33,866 | INFO | Moved: video.mp4 -> Videos/ [2026-05-02 23:08:33]


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: '\\tmp\\demo_organizer\\logs\\organizer_20260502_230833.log'

---
## SECTION 7 — GitHub Project: datetime_utils.py
### Project Deadline Tracker

In [28]:
import sys, logging
from datetime import datetime, timedelta

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

TASKS = [
    ('Set up project repo',           '2024-12-01', 'HIGH'),
    ('Complete Day 21 notes',          '2025-01-15', 'HIGH'),
    ('Build file organizer tool',      '2025-01-20', 'MEDIUM'),
    ('Practice LeetCode — Week 3',     '2025-02-01', 'MEDIUM'),
    ('Mock interview session',         '2025-02-15', 'HIGH'),
    ('Submit job applications — Q1',   '2025-03-01', 'HIGH'),
    ('Build GenAI project portfolio',  '2025-04-01', 'LOW'),
    ('Complete 30-day challenge',      '2025-05-01', 'HIGH'),
]

def parse_deadline(date_str: str) -> datetime:
    return datetime.strptime(date_str, '%Y-%m-%d').replace(hour=23, minute=59, second=59)

def days_remaining(deadline: datetime) -> int:
    return (deadline - datetime.now()).days

def urgency_label(days: int) -> str:
    if days < 0:   return "OVERDUE"
    if days == 0:  return "DUE TODAY"
    if days <= 3:  return "CRITICAL"
    if days <= 7:  return "URGENT"
    if days <= 14: return "UPCOMING"
    return "FUTURE"

def print_task_report(tasks: list) -> None:
    now = datetime.now()
    logging.info("=" * 75)
    logging.info(f"DEADLINE TRACKER — {now.strftime('%A, %B %d %Y at %H:%M:%S')}")
    logging.info("=" * 75)
    logging.info(f"{'Task':<35} {'Deadline':<12} {'Days':<6} {'Priority':<8} Status")
    logging.info("-" * 75)
    overdue, critical = [], []
    for name, date_str, priority in tasks:
        deadline = parse_deadline(date_str)
        days = days_remaining(deadline)
        label = urgency_label(days)
        days_display = f"-{abs(days)}" if days < 0 else str(days)
        formatted_date = deadline.strftime('%d %b %Y')
        logging.info(f"{name:<35} {formatted_date:<12} {days_display:<6} {priority:<8} [{label}]")
        if days < 0:    overdue.append((name, abs(days)))
        elif days <= 3: critical.append((name, days))
    logging.info("=" * 75)
    if overdue:
        logging.info(f"OVERDUE ({len(overdue)}):")
        for name, d in sorted(overdue, key=lambda x: -x[1]):
            logging.info(f"  {name} — {d} days ago")
    if critical:
        logging.info(f"CRITICAL — due within 3 days ({len(critical)}):")
        for name, d in sorted(critical, key=lambda x: x[1]):
            logging.info(f"  {name} — {d} days left")

def demo_timedelta_arithmetic() -> None:
    now = datetime.now()
    logging.info("\n--- timedelta Arithmetic Demos ---")
    q2_start = datetime(now.year, 4, 1)
    q3_start = datetime(now.year, 7, 1)
    logging.info(f"Q2 duration: {(q3_start - q2_start).days} days")
    hundred_days = now + timedelta(days=100)
    logging.info(f"100 days from now: {hundred_days.strftime('%A, %B %d %Y')}")
    year_end   = datetime(now.year, 12, 31)
    year_start = datetime(now.year,  1,  1)
    weeks = (year_end - year_start).days / 7
    logging.info(f"Weeks in {now.year}: {weeks:.1f}")
    python_release = datetime(1991, 2, 20)
    python_age = now - python_release
    logging.info(f"Python has existed for {python_age.days:,} days")
    logging.info(f"That is {python_age.total_seconds():,.0f} seconds total")

print_task_report(TASKS)
demo_timedelta_arithmetic()

2026-05-02 23:08:34,306 | ===========================================================================
2026-05-02 23:08:34,307 | DEADLINE TRACKER — Saturday, May 02 2026 at 23:08:34
2026-05-02 23:08:34,308 | ===========================================================================
2026-05-02 23:08:34,308 | Task                                Deadline     Days   Priority Status
2026-05-02 23:08:34,309 | ---------------------------------------------------------------------------
2026-05-02 23:08:34,310 | Set up project repo                 01 Dec 2024  -517   HIGH     [OVERDUE]
2026-05-02 23:08:34,311 | Complete Day 21 notes               15 Jan 2025  -472   HIGH     [OVERDUE]
2026-05-02 23:08:34,311 | Build file organizer tool           20 Jan 2025  -467   MEDIUM   [OVERDUE]
2026-05-02 23:08:34,312 | Practice LeetCode — Week 3          01 Feb 2025  -455   MEDIUM   [OVERDUE]
2026-05-02 23:08:34,313 | Mock interview session              15 Feb 2025  -441   HIGH     [OVERDUE]
2026-05-02 2

---
## SECTION 8 — Interview Gotchas and Q&A

In [29]:
import os, sys, time
from pathlib import Path
from datetime import datetime, timedelta, timezone

print("GOTCHA 1: os.path.join discards everything before an absolute component")
result = os.path.join('/home/user', '/tmp', 'file.txt')
print("  os.path.join('/home/user', '/tmp', 'file.txt') =", result)   # /tmp/file.txt !
print("  pathlib same:", Path('/home/user') / Path('/tmp') / 'file.txt')  # /tmp/file.txt
print()

print("GOTCHA 2: timedelta.seconds vs total_seconds()")
d = timedelta(days=1, seconds=30)
print("  timedelta(days=1, seconds=30)")
print("  d.seconds:", d.seconds)                   # 30
print("  d.total_seconds():", d.total_seconds())   # 86430.0
print()

print("GOTCHA 3: Cannot compare naive and aware datetimes")
naive = datetime.now()
aware = datetime.now(timezone.utc)
try:
    diff = aware - naive
except TypeError as e:
    print("  TypeError:", e)
print()

print("GOTCHA 4: time.time() can go backward, perf_counter() cannot")
print("  time() adjustable:", time.get_clock_info('time').adjustable)
print("  perf_counter adjustable:", time.get_clock_info('perf_counter').adjustable)
print()

print("GOTCHA 5: sys.getsizeof is SHALLOW")
big_list = list(range(1000))
print("  getsizeof of list with 1000 ints:", sys.getsizeof(big_list), "bytes")
print("  This does NOT include the integers themselves, only the container")
print()

print("GOTCHA 6: pathlib glob returns a generator — consumed only ONCE")
gen = Path('/tmp').glob('*')
count1 = len(list(gen))
count2 = len(list(gen))   # generator exhausted
print(f"  First iteration: {count1} items")
print(f"  Second iteration: {count2} items  (generator is exhausted)")
print()

print("GOTCHA 7: strptime raises ValueError on bad input, not None")
try:
    bad = datetime.strptime('not-a-date', '%Y-%m-%d')
except ValueError as e:
    print("  ValueError:", e)

GOTCHA 1: os.path.join discards everything before an absolute component
  os.path.join('/home/user', '/tmp', 'file.txt') = /tmp\file.txt
  pathlib same: \tmp\file.txt

GOTCHA 2: timedelta.seconds vs total_seconds()
  timedelta(days=1, seconds=30)
  d.seconds: 30
  d.total_seconds(): 86430.0

GOTCHA 3: Cannot compare naive and aware datetimes
  TypeError: can't subtract offset-naive and offset-aware datetimes

GOTCHA 4: time.time() can go backward, perf_counter() cannot
  time() adjustable: True
  perf_counter adjustable: False

GOTCHA 5: sys.getsizeof is SHALLOW
  getsizeof of list with 1000 ints: 8056 bytes
  This does NOT include the integers themselves, only the container

GOTCHA 6: pathlib glob returns a generator — consumed only ONCE
  First iteration: 1 items
  Second iteration: 0 items  (generator is exhausted)

GOTCHA 7: strptime raises ValueError on bad input, not None
  ValueError: time data 'not-a-date' does not match format '%Y-%m-%d'


In [30]:
print("COMMON INTERVIEW QUESTIONS AND ANSWERS")
print("=" * 60)

qa = [
    ("Q1: Difference between os.mkdir and os.makedirs?",
     "mkdir creates ONE directory. Fails if parent does not exist.\n"
     "     makedirs creates the full path including all missing parents.\n"
     "     Use exist_ok=True to avoid error if directory already exists."),
    ("Q2: Difference between os.path and pathlib.Path?",
     "os.path is a module of string functions — paths are plain strings.\n"
     "     pathlib.Path is an object with methods and properties.\n"
     "     pathlib is the modern approach (Python 3.4+). os.path still works."),
    ("Q3: What is sys.argv[0]?",
     "Always the script name/path. User args start at sys.argv[1]."),
    ("Q4: When to use perf_counter vs time()?",
     "perf_counter: benchmarking code, measuring elapsed time.\n"
     "     time(): real-world timestamps, recording when events happened.\n"
     "     time() can go backward. perf_counter cannot."),
    ("Q5: What is the difference between strptime and strftime?",
     "strptime = string PARSE to datetime.  (p = parse, input is string)\n"
     "     strftime = string FORMAT from datetime. (f = format, output is string)"),
    ("Q6: How do you do recursive file search?",
     "Option 1: os.walk — yields (dirpath, dirnames, filenames) tuples.\n"
     "     Option 2: Path.rglob('*.py') — generator of matching Path objects.\n"
     "     Option 3: Path.glob('**/*.py') — same with explicit pattern."),
    ("Q7: Why avoid os.system? What to use instead?",
     "os.system passes command to the shell — shell injection risk.\n"
     "     Use subprocess.run(['cmd', 'arg']) — args as list, no shell, safer."),
    ("Q8: What is the difference between timedelta.seconds and total_seconds()?",
     "timedelta(days=1, seconds=30).seconds = 30 (only the seconds component)\n"
     "     timedelta(days=1, seconds=30).total_seconds() = 86430.0 (entire duration)"),
]

for question, answer in qa:
    print(f"\n{question}")
    print(f"     {answer}")

COMMON INTERVIEW QUESTIONS AND ANSWERS

Q1: Difference between os.mkdir and os.makedirs?
     mkdir creates ONE directory. Fails if parent does not exist.
     makedirs creates the full path including all missing parents.
     Use exist_ok=True to avoid error if directory already exists.

Q2: Difference between os.path and pathlib.Path?
     os.path is a module of string functions — paths are plain strings.
     pathlib.Path is an object with methods and properties.
     pathlib is the modern approach (Python 3.4+). os.path still works.

Q3: What is sys.argv[0]?
     Always the script name/path. User args start at sys.argv[1].

Q4: When to use perf_counter vs time()?
     perf_counter: benchmarking code, measuring elapsed time.
     time(): real-world timestamps, recording when events happened.
     time() can go backward. perf_counter cannot.

Q5: What is the difference between strptime and strftime?
     strptime = string PARSE to datetime.  (p = parse, input is string)
     strftime

---
## Quick Reference Summary

In [31]:
summary = '''
==========================================================================
DAY 21 QUICK REFERENCE
==========================================================================

OS MODULE
---------
os.getcwd()                    Current directory as string
os.chdir(path)                 Change current directory
os.listdir(path)               List names in directory (non-recursive)
os.mkdir(path)                 Create one directory
os.makedirs(path, exist_ok)    Create full nested path
os.remove(path)                Delete a file
os.rmdir(path)                 Delete empty directory
os.rename(src, dst)            Move or rename file/directory
os.path.exists(p)              Does path exist?
os.path.isfile(p)              Is it a regular file?
os.path.isdir(p)               Is it a directory?
os.path.join(a, b)             Join path parts using OS separator
os.path.basename(p)            Last component of path
os.path.dirname(p)             Parent path
os.path.splitext(p)            (root, extension) tuple
os.path.abspath(p)             Absolute path from cwd
os.environ.get(k, default)     Read environment variable safely
os.walk(top)                   Recursive traversal generator
os.getpid()                    Current process ID
os.cpu_count()                 Number of logical CPUs

SYS MODULE
----------
sys.argv                       List of command line arguments
sys.path                       List of import search directories
sys.exit(code)                 Exit process (raises SystemExit)
sys.stdout.write(s)            Write to standard output
sys.stderr.write(s)            Write to standard error
sys.version                    Python version string
sys.platform                   'linux', 'darwin', 'win32'
sys.maxsize                    Largest int for this platform
sys.getsizeof(obj)             Shallow size in bytes
sys.modules                    Dict of all imported modules

PATHLIB MODULE
--------------
Path.cwd()                     Current directory as Path object
Path.home()                    User home directory
p / 'sub' / 'file.txt'         Join using / operator (clean syntax)
p.joinpath(a, b)               Same as / operator, method style
p.name                         Filename with extension
p.stem                         Filename without extension
p.suffix                       Extension including dot (.py)
p.parent                       Containing directory (one level up)
p.parts                        Tuple of all path components
p.exists()                     Does path exist?
p.is_file() / p.is_dir()       File or directory check
p.is_absolute()                Is path absolute?
p.mkdir(parents, exist_ok)     Create directory
p.touch()                      Create empty file or update mtime
p.unlink()                     Delete file
p.rmdir()                      Delete empty directory
p.rename(target)               Move or rename
p.read_text(encoding)          Read entire file as string
p.write_text(s, encoding)      Write string to file
p.read_bytes()                 Read file as raw bytes
p.write_bytes(b)               Write raw bytes to file
p.glob('*.py')                 Match files in this directory only
p.rglob('*.py')                Recursive match all subdirectories
p.iterdir()                    Iterate direct children
p.resolve()                    Absolute path, resolves symlinks
p.as_posix()                   String with forward slashes (cross-platform)

DATETIME MODULE
---------------
date(y, m, d)                  Create date object
date.today()                   Today's date
datetime(y, m, d, H, M, S)     Create datetime object
datetime.now()                 Current datetime (naive, local time)
datetime.now(timezone.utc)     Current UTC datetime (timezone-aware)
dt.strftime('%Y-%m-%d')        Format datetime to string
datetime.strptime(s, fmt)      Parse string to datetime
dt.isoformat()                 Return 'YYYY-MM-DDTHH:MM:SS' string
datetime.fromisoformat(s)      Parse ISO 8601 string
timedelta(days=N)              Create a duration object
dt + timedelta(days=7)         Compute future date
dt2 - dt1                      Difference as timedelta
delta.days                     Days component of timedelta
delta.total_seconds()          TOTAL seconds (not just component)
dt.timestamp()                 Convert to Unix timestamp float
datetime.fromtimestamp(ts)     Create datetime from Unix timestamp
timezone.utc                   Built-in UTC timezone object
timezone(timedelta(hours=5))   Custom timezone offset
dt.astimezone(tz)              Convert between timezones

TIME MODULE
-----------
time.time()                    Unix timestamp — can go backward
time.perf_counter()            High-res benchmark clock
time.monotonic()               Never decreases — use for timeouts
time.process_time()            CPU time only — excludes sleep
time.sleep(seconds)            Pause execution (releases GIL)
time.localtime(ts)             Unix timestamp to local struct_time
time.gmtime(ts)                Unix timestamp to UTC struct_time
time.strftime(fmt, struct)     Format struct_time to string
time.mktime(struct)            Local struct_time to Unix timestamp
time.get_clock_info(name)      Metadata about a named clock
time.timezone                  UTC offset of local timezone in seconds
time.tzname                    ('Standard', 'DST') timezone name tuple

==========================================================================
'''
print(summary)


DAY 21 QUICK REFERENCE

OS MODULE
---------
os.getcwd()                    Current directory as string
os.chdir(path)                 Change current directory
os.listdir(path)               List names in directory (non-recursive)
os.mkdir(path)                 Create one directory
os.makedirs(path, exist_ok)    Create full nested path
os.remove(path)                Delete a file
os.rmdir(path)                 Delete empty directory
os.rename(src, dst)            Move or rename file/directory
os.path.exists(p)              Does path exist?
os.path.isfile(p)              Is it a regular file?
os.path.isdir(p)               Is it a directory?
os.path.join(a, b)             Join path parts using OS separator
os.path.basename(p)            Last component of path
os.path.dirname(p)             Parent path
os.path.splitext(p)            (root, extension) tuple
os.path.abspath(p)             Absolute path from cwd
os.environ.get(k, default)     Read environment variable safely
os.walk(top)   